In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import sys 
sys.path.append("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline")
from config.constants import MIMIC_IV_PATH
%load_ext autoreload
%autoreload 2

# Check Sina's splits

In [ ]:
import zipfile, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import glob
REPO_ROOT = Path.cwd().parent

base        = REPO_ROOT / "saved_data"
folds_dir   = base / "folds"
cohorts_dir = base / "cohorts" / "DTB"
zip_path    = base / "folds_Sina" / "split_ids.zip"
mimic_pts   = pd.read_csv(Path(MIMIC_IV_PATH) / "hosp" / "patients.csv.gz")



def load_sina(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        return {n: set(int(l) for l in zf.read(f"split_ids/{n}.txt").decode().splitlines() if l.strip())
                for n in ["train", "tuning", "held_out"]}
        
sina = load_sina(zip_path)

train,tuning,held_out = sina['train'],sina['tuning'],sina['held_out']
print("all mimic patients count:",mimic_pts.subject_id.nunique())
print("patients in Sina's splits:",len(train) + len(tuning) + len(held_out))
print("len training",len(train),"len validation",len(tuning),"len test",len(held_out))

In [ ]:
# check cohorts to see if Sina's cohorts are usable
files = sorted(glob.glob(str(cohorts_dir / "cohort_*.csv.gz")))
records = []
all_cohort_pats = set()
for f in files:#[:10]:
    cohort_name = Path(f).name.replace("cohort_", "").replace(".csv.gz", "")
    df = pd.read_csv(f, usecols=["subject_id", "hadm_id"])
    pats = set(df.subject_id.unique().tolist())
    all_cohort_pats |= pats

    n = len(pats)
    n_a = df.hadm_id.nunique()
    n_train = len(pats & sina["train"])
    n_val   = len(pats & sina["tuning"])
    n_test  = len(pats & sina["held_out"])
    n_uncovered = n - n_train - n_val - n_test

    records.append({
        "cohort": cohort_name,
        "n": n,
        "n_a": n_a,
        "n_train": n_train,
        "n_val": n_val,
        "n_test": n_test,
        "n_uncovered": n_uncovered,
    })

df_sina = pd.DataFrame(records).sort_values("n_test")

all_mimic = set(mimic_pts.subject_id.tolist())
not_in_cohort = all_mimic - all_cohort_pats

print(f"total cohorts        : {len(df_sina)}")
print(f"any uncovered pts    : {(df_sina.n_uncovered > 0).sum()}")
print(f"all MIMIC patients   : {len(all_mimic)}")
print(f"in some cohort       : {len(all_cohort_pats)}")
print(f"NOT in any cohort    : {len(not_in_cohort)} ({100*len(not_in_cohort)/len(all_mimic):.1f}%)")


In [ ]:
import numpy as np
import pandas as pd

BANDS  = [0, 1, 2, 3, 5, 10, np.inf]
LABELS = ["≤1%", "≤2%", "≤3%", "≤5%", "≤10%", ">10%"]

def add_deviation(d):
    """Add split percentages, max deviation from 80/10/10, and its deviation band."""
    d = d.assign(
        train_pct=100 * d.n_train / d.n,
        val_pct=100 * d.n_val / d.n,
        test_pct=100 * d.n_test / d.n,
    )
    d = d.assign(max_dev=np.maximum.reduce([
        (d.train_pct - 80).abs(),
        (d.val_pct - 10).abs(),
        (d.test_pct - 10).abs(),
    ]))
    d = d.assign(band=pd.cut(d.max_dev, bins=BANDS, labels=LABELS, include_lowest=True))
    return d


In [ ]:

def plot_split(d, title="", xlim=(60, 100), ylim=(0, 25)):
    """Plot deviation bands and train%-vs-test% scatter (expects add_deviation columns)."""
    band_counts = d.band.value_counts().reindex(LABELS, fill_value=0)

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].bar(band_counts.index, band_counts.values, color="#4c72b0")
    ax[0].set(title=f"{title}: deviation from 80/10/10", xlabel="max abs deviation", ylabel="cohorts")
    for i, c in enumerate(band_counts.values):
        ax[0].text(i, c + 0.5, str(c), ha="center")

    sc = ax[1].scatter(d.train_pct, d.test_pct, c=d.n, cmap="viridis", norm=LogNorm(), s=25)
    ax[1].scatter([80], [10], color="red", marker="*", s=300, label="target 80/10")
    ax[1].set(title=f"{title}: train% vs test% (color = n)", xlabel="train %", ylabel="test %")
    ax[1].set_xlim(*xlim); ax[1].set_ylim(*ylim)
    ax[1].legend()
    fig.colorbar(sc, ax=ax[1], label="cohort size (n)")
    fig.tight_layout()
    plt.show()


In [ ]:
df_sina = add_deviation(df_sina)
plot_split(df_sina, title="Sina splits")

# Optimization

In [ ]:
#Goal: one global 80/10/10 patient split so a single pretrained model is leakage-free for all 4,648 cohorts (47,664 patients) — instead of one model per cohort.
#Objective: assign each patient to exactly one group, minimizing total per-cohort deviation from 80/10/10.
#Baseline (Sina's split): covers all patients, but small cohorts drift far from 80/10/10.
#Tried (Approach A): exact ILP with min-size constraints via Gurobi (~143k binary vars) → intractable (hours, ~50% gap) and distorts small cohorts.
#Chose (Approach B): pure minimum-error heuristic (local search) — no constraints, no solver; drop small cohorts post-hoc (keep ≥10 test patients).
#Result: obj 35.12; 4,112/4,648 cohorts within 1% of target; avg deviation <1%

In [ ]:
from flab_cohorts.utils.split_optimization import SplitOptimizer

opt = SplitOptimizer(cohorts_dir=cohorts_dir, ratios=(0.8, 0.1, 0.1), seed=42)

opt.load_cohorts()
opt.build_index()
opt.heuristic(patience=10, tol=1e-6, max_passes=200)

all_mimic = set(mimic_pts.subject_id.tolist())
opt.save_global_split(base / "folds_global" / "split_ids", all_mimic)

df_opt = opt.summary()


In [ ]:
df_opt = add_deviation(df_opt)
plot_split(df_opt, title="Optimized")

In [ ]:
# Repeat across seeds to confirm the objective is stable (robust to initialization).
objs = []
for seed in range(5):
    o = SplitOptimizer(cohorts_dir=cohorts_dir, seed=seed)
    o.load_cohorts(); o.build_index(); o.heuristic()
    objs.append(o._ratio_obj(o.count, o.n).sum())
print("objectives across seeds:", [round(x, 3) for x in objs])



In [ ]:
# Look into the cohorts which with higher error rates

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

thresholds = [20, 30, 50, 100]

df = df_opt

fig, ax = plt.subplots(figsize=(9, 5))
bins = np.logspace(np.log10(df.n.min()), np.log10(df.n.max()), 50)
ax.hist(df.n, bins=bins, color="#4c72b0")
ax.set_xscale("log")
ax.set(title="Cohort size distribution", xlabel="patients per cohort (n)", ylabel="cohorts")

for thr in thresholds:
    below = int((df.n < thr).sum())
    ax.axvline(thr, color="red", ls="--", alpha=0.7)
    ax.text(thr, ax.get_ylim()[1]*0.95, f"n<{thr}: {below}", rotation=90,
            va="top", ha="right", fontsize=9, color="red")

fig.tight_layout()
plt.show()

for thr in thresholds:
    print(f"cohorts with n < {thr:>4}: {(df.n < thr).sum():>4}  /  {len(df)}")

## chekc the new splits

In [ ]:
global_dir = base / "folds_global" / "split_ids"

def load_split(split_dir):
    return {
        key: set(int(l) for l in (split_dir / f"{key}.txt").read_text().splitlines() if l.strip())
        for key in ["train", "tuning", "held_out"]
    }

g = load_split(global_dir)
train_g, tuning_g, held_out_g = g["train"], g["tuning"], g["held_out"]

tot = len(train_g) + len(tuning_g) + len(held_out_g)
print(f"train   : {len(train_g):>7} ({100*len(train_g)/tot:.2f}%)")
print(f"tuning  : {len(tuning_g):>7} ({100*len(tuning_g)/tot:.2f}%)")
print(f"held_out: {len(held_out_g):>7} ({100*len(held_out_g)/tot:.2f}%)")
print("total   :", tot)
print("unique  :", len(train_g | tuning_g | held_out_g))

assert not (train_g & held_out_g)
assert not (train_g & tuning_g)
assert not (tuning_g & held_out_g)
print("OK: no patient in more than one split")



In [ ]:
g    = load_split(base / "folds_global" / "split_ids")     # my global split
sina = load_sina(base / "folds_Sina" / "split_ids.zip")    # Sina's split

# 1) global sizes / ratios
def ratios(d):
    tot = sum(len(v) for v in d.values())
    return {k: f"{len(v)} ({100*len(v)/tot:.2f}%)" for k, v in d.items()} | {"total": tot}
print("global:", ratios(g))
print("sina  :", ratios(sina))

# 2) how many patients changed group (over the shared patients)
lbl_g    = {p: k for k, v in g.items()    for p in v}
lbl_sina = {p: k for k, v in sina.items() for p in v}
shared = set(lbl_g) & set(lbl_sina)
moved = sum(lbl_g[p] != lbl_sina[p] for p in shared)
print(f"\nshared patients: {len(shared)}, changed group: {moved} ({100*moved/len(shared):.2f}%)")
